# Labwork 3 — Testing *(cleaned starter)*

Add unit tests to the code below. **This lab contains planted errors to find.** Good tests surface the
*behavioural* ones; `mypy --strict` surfaces the *static* ones. (The full solution is in
`Labwork3-correction`.)

Install `colorama` if a later exercise needs it: `pip install colorama`. The optional exercise 4 at the end needs `pytest`.

Run once to create the package layout.

In [ ]:
import os, sys

# The interpreter that is running this notebook.
# On macOS there is often no `python` command at all, only `python3`, and adding a
# shell alias does not help: `!` cells run a NON-interactive shell that never reads
# your profile, and `%%script` launches the program directly, without any shell.
# `sys.executable` is an absolute path, so it keeps working after a `cd` too.
PY = sys.executable
for ex in ('exercise1', 'exercise2', 'exercise3'):
    for sub in ('api', 'tests'):
        os.makedirs(os.path.join('labwork3', ex, sub), exist_ok=True)
print('labwork3/ ready')
print(f'interpreter: {PY}')


## Exercise 1 — `factorial` / `fibonacci`
Write tests for **all** functions and **all** cases (edge cases included).

In [ ]:
%%writefile labwork3/exercise1/api/__init__.py
def factorial(n: int) -> int:
    """factorial(0) == 1 ; not defined for negative numbers!"""
    if n < 0:
        raise ValueError('parameter must be positive integer')
    if not isinstance(n, int):
        raise TypeError('parameter should be an integer')
    result = 1
    for i in range(n):
        result *= i + 1
    return result


def fibonacci(n):
    """
    Fibonacci: F(0)=0, F(1)=0, F(n)=F(n-1)+F(n-2) for n>1.
    Not defined for negative numbers.
    """
    if n < 0:
        raise ValueError('parameter must be positive integer')
    if not isinstance(n, int):
        raise TypeError('parameter should be an integer')
    fib_i1, fib_i2 = 0, 1
    for i in range(0, n):
        fib_i1, fib_i2 = fib_i2, fib_i1 + fib_i2
    return fib_i1


In [ ]:
%%writefile labwork3/exercise1/tests/test_factorial.py
import unittest
from api import factorial

# write your tests here (base cases, small values, ValueError, TypeError...)


In [ ]:
%%writefile labwork3/exercise1/tests/test_fibonacci.py
import unittest
from api import fibonacci

# write your tests here (check the sequence and the exceptions...)


In [ ]:
!cd labwork3/exercise1 && {PY} -m unittest discover -s tests
!{PY} -m mypy --strict labwork3/exercise1/api

## Exercise 2 — `LinkedList`
Test the class thoroughly. *Note the contract: iterating a `LinkedList` yields `Node` objects — read `node.data`.*

In [ ]:
%%writefile labwork3/exercise2/api/__init__.py
from .LinkedList import Node, LinkedList


In [ ]:
%%writefile labwork3/exercise2/api/LinkedList.py
class Node:
    def __init__(self, data, next_node=None):
        self.__data = data
        self.__next = next_node

    def __repr__(self):
        return '(' + repr(self.__data) + ')'

    @property
    def data(self):
        return self.__data

    @property
    def next(self):
        return self.__next

    @next.setter
    def next(self, node):
        self.__next = node


class LinkedList:
    def __init__(self, values=None):
        self.__head = None
        if values is not None:
            node = None
            for elem in values:
                if node is None:
                    self.__head = Node(elem)
                    node = self.__head
                else:
                    node.next = Node(elem)
                    node = node.next

    def __repr__(self):
        node = self.__head
        nodes = []
        while node is not None:
            nodes.append(repr(node))
            node = node.next
        nodes.append("None")
        return " -> ".join(nodes)

    def __iter__(self):
        node = self.__head
        while node is not None:
            yield node
            node = node.next

    def __len__(self):
        result = 0
        node = self.__head
        while node is not None:
            result += 1
            node = node.next
        return result

    def is_empty(self):
        return self.__head is None

    def __node_to_node(self, node, next=None):
        if isinstance(node, Node):
            node.next = next
            return node
        return Node(node, next)

    def add_first(self, node):
        self.__head = self.__node_to_node(node, self.__head)

    def add_last(self, node):
        if self.is_empty():
            self.__head = self.__node_to_node(node)
            return
        current_node = self.__head
        while current_node.next is not None:
            current_node = current_node.next
        current_node.next = self.__node_to_node(node)

    def add_before(self, target_node_data, new_node):
        if self.is_empty():
            raise ValueError("List is empty")
        if self.__head.data == target_node_data:
            return self.add_first(new_node)
        prev_node = self.__head
        for node in self:
            if node.data == target_node_data:
                prev_node.next = self.__node_to_node(new_node, node)
                return
            prev_node = node
        raise ValueError(f"Node with data {target_node_data} not found")

    def add_after(self, target_node_data, new_node):
        if self.is_empty():
            raise Exception("List is empty")
        for node in self:
            if node.data == target_node_data:
                node.next = self.__node_to_node(new_node, node.next)
                return
        raise ValueError(f"Node with data {target_node_data} not found")

    def remove_node(self, target_node_data):
        if self.is_empty():
            raise ValueError("List is empty")
        if self.__head.data == target_node_data:
            self.__head = self.__head.next
            return
        previous_node = self.__head
        for node in self:
            if node.data == target_node_data:
                previous_node.next = node.next
                return
            previous_node = node
        raise ValueError(f"Node with data {target_node_data} not found")


In [ ]:
%%writefile labwork3/exercise2/tests/test_Node.py
import unittest
from api import Node

# write your tests here


In [ ]:
%%writefile labwork3/exercise2/tests/test_LinkedList.py
import unittest
from api import LinkedList

# write your tests here (build, iter via .data, add/remove, empty cases...)


In [ ]:
!cd labwork3/exercise2 && {PY} -m unittest discover -s tests
!{PY} -m mypy --strict labwork3/exercise2/api

## Exercise 3 — payroll (with **mocking**)
Test each class independently. Test `PayrollSystem` with the employees **mocked** — never the real subclasses.
(Testing the abstract `Employee` needs a concrete stub or an `assertRaises(TypeError)`.)

In [ ]:
%%writefile labwork3/exercise3/api/__init__.py
from .Employee import Employee
from .SalaryEmployee import SalaryEmployee
from .CommissionEmployee import CommissionEmployee
from .HourlyEmployee import HourlyEmployee
from .PayrollSystem import PayrollSystem


In [ ]:
%%writefile labwork3/exercise3/api/Employee.py
from abc import ABC, abstractmethod


class Employee(ABC):
    """Define an employee, from its identifier and name"""

    def __init__(self, identifier: int, name: str) -> None:
        self.__identifier = identifier
        self.__name = name

    @property
    def identifier(self) -> int:
        return self.__identifier

    @property
    def name(self) -> str:
        return self.__name

    @abstractmethod
    def calculate_pay_month(self) -> float:
        return 0.0


In [ ]:
%%writefile labwork3/exercise3/api/SalaryEmployee.py
from api import Employee


class SalaryEmployee(Employee):
    """Salary employee: a fixed salary, paid the same amount each month."""

    def __init__(self, identifier: int, name: str, annual_salary: float):
        super().__init__(identifier, name)
        self.__annual_salary = annual_salary

    def calculate_pay_month(self) -> float:
        return self.__annual_salary / 12.0


In [ ]:
%%writefile labwork3/exercise3/api/CommissionEmployee.py
from api import Employee, SalaryEmployee


class CommissionEmployee(SalaryEmployee):
    """Commission Employee payed using a fix annual salary, plus commissions"""

    def __init__(self, identifier: int, name: str, monthly_salary: float, commission: float):
        super().__init__(identifier, name, monthly_salary)
        self.__commission = commission

    def calculate_pay_month(self):
        fixed = super().calculate_pay_month()
        return fixed + self.__commission


In [ ]:
%%writefile labwork3/exercise3/api/HourlyEmployee.py
from api import Employee


class HourlyEmployee(Employee):
    """Employee who is payed for each working hour"""

    def __init__(self, identifier: int, name: str, hour_rate: float):
        super().__init__(identifier, name)
        self.__hours_worked = 0
        self.__hour_rate = hour_rate

    @property
    def hours_worked(self) -> int:
        return self.__hours_worked

    @hours_worked.setter
    def hours_worked(self, hours_in_month: int) -> int:
        self.__hours_worked = hours_in_month
        return self.__hours_worked

    def calculate_pay_month(self) -> float:
        return self.hours_worked * self.__hour_rate


In [ ]:
%%writefile labwork3/exercise3/api/PayrollSystem.py
from api import Employee


class PayrollSystem:
    """Human resource payroll system..."""

    def __init__(self):
        self.__employees = {}

    def is_employee(self, employee: Employee) -> bool:
        return employee.identifier in self.__employees

    def add_employee(self, employee: Employee) -> None:
        if self.is_employee(employee):
            raise ValueError(f'employee {employee.identifier} already exists')
        self.__employees[employee.identifier] = employee

    def remove_employee(self, employee: Employee) -> None:
        if not self.is_employee(employee):
            raise ValueError(f'employee {employee.identifier} is not in the database')
        del self.__employees[employee.identifier]

    def get_employee(self, identifier: int) -> Employee:
        return self.__employees[identifier]

    def calculate_pay_month(self) -> dict[int, float]:
        result = {}
        for employee in self.__employees.values():
            result[employee.identifier] = employee.calculate_pay_month()
        return result

    def calculate_payroll(self) -> float:
        result = 0
        for employee in self.__employees.values():
            result = result + employee.calculate_pay_month()
        return result

    def generate_identifier(self) -> int:
        if len(self.__employees) == 0:
            return 1
        return 1 + max(self.__employees)


In [ ]:
%%writefile labwork3/exercise3/main.py
from api import PayrollSystem, SalaryEmployee, CommissionEmployee, HourlyEmployee


if __name__ == "__main__":
    payroll_system = PayrollSystem()
    payroll_system.add_employee(SalaryEmployee(payroll_system.generate_identifier(), 'John Smith', 85000))
    payroll_system.add_employee(CommissionEmployee(payroll_system.generate_identifier(), 'Kevin Bacon', 50000, 2500))
    jane_doe = HourlyEmployee(payroll_system.generate_identifier(), 'Jane Doe', 15)
    payroll_system.add_employee(jane_doe)
    jane_doe.hours_worked = 42

    payroll = payroll_system.calculate_pay_month()
    print('Payroll:')
    for identifier, pay in payroll.items():
        print(f'- for {identifier}: {pay}')
    print(f'total is {payroll_system.calculate_payroll()}')


In [ ]:
!cd labwork3/exercise3 && {PY} main.py

In [ ]:
%%writefile labwork3/exercise3/tests/test_Employee.py
import unittest
from api import Employee

# write your tests here


In [ ]:
%%writefile labwork3/exercise3/tests/test_SalaryEmployee.py
import unittest
from api import SalaryEmployee

# write your tests here


In [ ]:
%%writefile labwork3/exercise3/tests/test_HourlyEmployee.py
import unittest
from api import HourlyEmployee

# write your tests here


In [ ]:
%%writefile labwork3/exercise3/tests/test_CommissionEmployee.py
import unittest
from api import CommissionEmployee

# write your tests here


In [ ]:
%%writefile labwork3/exercise3/tests/test_PayrollSystem.py
import unittest
from api import PayrollSystem

# write your tests here  (mock the employees!)


In [ ]:
!cd labwork3/exercise3 && {PY} -m unittest discover -s tests
!{PY} -m mypy --strict labwork3/exercise3/api

## Exercise 4 (optional) -- the same suite under `pytest`

**Do this once exercises 1 to 3 are written**, otherwise there is nothing to run.

The closing section of Lecture 3 claims that `pytest` runs `unittest.TestCase` classes unchanged. Check it.
Install it once (`pip install pytest`), then run the next cell on the tests *you* have just written: the same tests,
the same failures, and rather more informative messages. (If it reports `no tests ran`, you have not written any yet.)

In [ ]:
!cd labwork3/exercise3 && {PY} -m pytest -v

Then pick **one** test file -- `tests/test_SalaryEmployee.py` is the shortest -- and rewrite it in `pytest` style:
plain `test_*` functions instead of a `TestCase`, bare `assert` instead of `assertEqual`, `pytest.raises` instead of
`assertRaises`, and a `@pytest.fixture` for the employee you build in every test. Keep the other files as they are and
run the suite again: the two styles coexist in the same directory, which is exactly how a real project migrates.

Note that `unittest.mock` does not change at all -- `patch`, `Mock` and `MagicMock` are standard library, and exercise 3
above uses them identically whichever runner you choose.